# Monitors & Reports

This file sets up the model, data, and infrastructure monitors. It also establishes a monitoring dashboard the code for generating reports on SageMaker.

Attribution: The code was made with the assistance of Perplexity.ai accessed in February 2026.

### Set Up Model & Baseline

In [1]:
#Import key libraries
#Sagemaker Imports
import sagemaker
from sagemaker import Session
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DatasetFormat,
    CronExpressionGenerator
)
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import IdentitySerializer
from sagemaker.deserializers import JSONDeserializer


#Other Imports
import s3fs
import pandas as pd
import boto3
import json
import time
import tarfile
import botocore.exceptions
import io
from datetime import datetime, timedelta
import numpy as np
from pathlib import Path

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
#Set up session
region = "us-east-1"
session = Session()
sm_client = boto3.client("sagemaker", region_name=region)
cw_client = boto3.client("cloudwatch", region_name=region)
s3_client = boto3.client("s3", region_name=region)

role = sagemaker.get_execution_role()  # or hard‑code your SageMaker role ARN
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"

In [3]:
# Config
bucket = "sagemaker-us-east-1-418418308994"
local_base = Path('/tmp/Models/benchmarks')
s3_base = f"s3://{bucket}/models/benchmarks"

print("🔍 Loading XGBoost artifacts...")

# XGBoost paths (prioritize local)
xgb_paths = {
    'local_tar.gz': local_base / 'xgboost/model.tar.gz',
    's3_tar.gz': f"{s3_base}/xgboost/model.tar.gz",
    's3_pkl': f"{s3_base}/xgboost/model.pkl",
    'local_metrics': local_base / 'xgboost/metrics.json',
    'local_model': local_base / 'xgboost/model.pkl'
}

# Auto-select best path
model_data = str(xgb_paths['local_tar.gz']) if xgb_paths['local_tar.gz'].exists() else xgb_paths['s3_tar.gz']
metrics_path = xgb_paths['local_metrics'] if xgb_paths['local_metrics'].exists() else None

print(f"Model data: {model_data}")
print(f"Metrics: {metrics_path}")
print("XGBoost paths ready!")

🔍 Loading XGBoost artifacts...
Model data: s3://sagemaker-us-east-1-418418308994/models/benchmarks/xgboost/model.tar.gz
Metrics: None
XGBoost paths ready!


In [34]:
# Get the latest XGBoost container for region
from sagemaker import image_uris
xgboost_container = image_uris.retrieve(
    framework="xgboost",
    region="us-east-1",
    version="1.7-1" 
)

print(f"Using XGBoost container: {xgboost_container}")

# Create model with updated container
new_xgb_endpoint_name = f"xgb-benchmark-endpoint-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

xg_model_new = Model(
    image_uri=xgboost_container,
    model_data="s3://sagemaker-us-east-1-418418308994/models/benchmarks/xgboost/model.tar.gz",
    role=role,
    sagemaker_session=session,
)

print(f"Deploying new endpoint: {new_xgb_endpoint_name}")

# Deploy with data capture
data_capture_prefix = f"{prefix}/data-capture"
data_capture_s3_uri = f"s3://{bucket}/{data_capture_prefix}"

xg_predictor_new = xg_model_new.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=new_xgb_endpoint_name,
    data_capture_config=sagemaker.model_monitor.DataCaptureConfig(
        enable_capture=True,
        sampling_percentage=100,
        destination_s3_uri=data_capture_s3_uri,
        capture_options=["REQUEST", "RESPONSE"],
    ),
)

print(f"Endpoint deployed: {new_xgb_endpoint_name}")

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Creating model with name: sagemaker-xgboost-2026-02-06-02-16-06-576


Using XGBoost container: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1
Deploying new endpoint: xgb-benchmark-endpoint-20260206-021606


INFO:sagemaker:Creating endpoint-config with name xgb-benchmark-endpoint-20260206-021606
INFO:sagemaker:Creating endpoint with name xgb-benchmark-endpoint-20260206-021606


------!✅ New endpoint deployed: xgb-benchmark-endpoint-20260206-021606


In [35]:
import time

# Wait for endpoint to be in service
print("Waiting for endpoint to be ready...")

sm_client = boto3.client('sagemaker', region_name='us-east-1')

while True:
    response = sm_client.describe_endpoint(EndpointName=new_xgb_endpoint_name)
    status = response['EndpointStatus']
    print(f"  Status: {status}")
    
    if status == 'InService':
        print("✅ Endpoint is ready!")
        break
    elif status == 'Failed':
        print("❌ Endpoint deployment failed!")
        print(f"Failure reason: {response.get('FailureReason', 'Unknown')}")
        break
    
    time.sleep(30)

Waiting for endpoint to be ready...
  Status: InService
✅ Endpoint is ready!


In [37]:
from sagemaker.predictor import Predictor

# Create predictor for the new endpoint
xg_predictor_new = Predictor(
    endpoint_name=new_xgb_endpoint_name,
    sagemaker_session=session,
)

# Load test data
baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv")
test_data = baseline_df.drop("target", axis=1).iloc[:5]

print(f"Test data shape: {test_data.shape}")

# Convert to CSV
csv_payload = test_data.to_csv(header=False, index=False)

# Send prediction
response = xg_predictor_new.predict(
    data=csv_payload,
    initial_args={
        'ContentType': 'text/csv',
        'Accept': 'text/csv'
    }
)

print(f"✅ Predictions: {response}")

Test data shape: (5, 20)
✅ Predictions: b'0.009759249165654182,0.010805039666593075,0.48084768652915955,0.03209933266043663,0.03439685329794884,0.4277508556842804,0.004340957384556532\n0.007653615437448025,0.007332483772188425,0.22988218069076538,0.07651730626821518,0.07866891473531723,0.5871987342834473,0.012746804393827915\n0.01511878240853548,0.01036760676652193,0.5404502153396606,0.03245336189866066,0.12612764537334442,0.2628888487815857,0.012593579478561878\n0.015056170523166656,0.004233742132782936,0.17002418637275696,0.01482530776411295,0.09608134627342224,0.6938716769218445,0.005907570943236351\n0.02310902625322342,0.016121942549943924,0.2598908543586731,0.01709647849202156,0.08900441229343414,0.5792668461799622,0.015510435216128826\n'


In [38]:
xgb_endpoint_name = "xgb-benchmark-endpoint-20260206-021606"

In [39]:
try:
    dq_stats_uri = dq_monitor.latest_baselining_job.baseline_statistics.file_name
    dq_constraints_uri = dq_monitor.latest_baselining_job.suggested_constraints.file_name
    print(f"Stats: {dq_stats_uri}")
    print(f"Constraints: {dq_constraints_uri}")
except:
    # List S3 output manually if SDK fails
    s3 = boto3.client('s3')
    response = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/baseline/")
    for obj in response.get('Contents', []):
        if obj['Key'].endswith('statistics.json'):
            dq_stats_uri = f"s3://{bucket}/{obj['Key']}"
        if obj['Key'].endswith('constraints.json'):
            dq_constraints_uri = f"s3://{bucket}/{obj['Key']}"
    print(f"Stats: {dq_stats_uri}")
    print(f"Constraints: {dq_constraints_uri}")

Stats: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json
Constraints: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json


In [40]:
#Find JSON files
s3 = boto3.client('s3')

response = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/baseline/")
files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.json')]
print("JSON files:")
for f in files:
    print(f"s3://{bucket}/{f}")

JSON files:
s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json
s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json


In [41]:
key = "models/benchmarks/monitoring/baseline/constraints.json"
obj = s3_client.get_object(Bucket=bucket, Key=key)
constraints = json.load(obj['Body'])

print("Full structure:")
print(json.dumps(constraints, indent=2)[:1000])  # First 1000 chars

constraints_df = pd.json_normalize(constraints['features'])
print("\nAvailable columns:")
print(constraints_df.columns.tolist())
print("\nFirst 5 rows:")
print(constraints_df.head())

Full structure:
{
  "version": 0.0,
  "features": [
    {
      "name": "meanfreq",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "sd",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "median",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "q25",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "q75",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": false
      }
    },
    {
      "name": "iqr",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints":

In [42]:
key = "models/benchmarks/monitoring/baseline/statistics.json"
obj = s3_client.get_object(Bucket=bucket, Key=key)
statistics = json.load(obj['Body'])

stats_df = pd.json_normalize(statistics['features'])
print("Audio Feature Statistics (Top 10):")
print(stats_df[['name', 
                'numerical_statistics.mean', 
                'numerical_statistics.std_dev', 
                'numerical_statistics.min', 
                'numerical_statistics.max',
                'numerical_statistics.completeness']].head(10))

Audio Feature Statistics (Top 10):
       name  numerical_statistics.mean  numerical_statistics.std_dev  \
0  meanfreq                2348.249412                   1468.419325   
1        sd                2447.622411                   1105.571653   
2    median                4593.042497                   2696.537284   
3       q25                   0.090211                      0.045230   
4       q75                -404.312476                    106.927208   
5       iqr                  87.612007                     25.563514   
6      skew                  19.789872                     22.195497   
7      kurt                  17.906020                     11.754694   
8    sp_ent                   3.565427                     10.358779   
9       sfm                   1.530392                      7.320638   

   numerical_statistics.min  numerical_statistics.max  \
0                  0.000000               7655.335725   
1                  0.000000               6324.351767   
2

### Schedule a data quality monitoring job for each endpoint

In [43]:
#Ensure monitor isn't already in place
try:
    dq_monitor.delete_monitoring_schedule(schedule_name_xgb_dq)
    print(f"Deleted existing: {schedule_name_xgb_dq}")
except:
    print("No existing schedule")

No existing schedule


In [44]:
#Create schedule
schedule_name_xgb_dq = "xgb-data-quality-schedule"

dq_monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name_xgb_dq,
    endpoint_input=xgb_endpoint_name,
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/data-quality/xgb",
    statistics="s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json",
    constraints="s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json",
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,
)

print(f"Hourly schedule live: {schedule_name_xgb_dq}")

INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-data-quality-schedule
ERROR:sagemaker.model_monitor.model_monitoring:Failed to create monitoring schedule.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/sagemaker/model_monitor/model_monitoring.py", line 2050, in create_monitoring_schedule
    self._create_monitoring_schedule_from_job_definition(
  File "/opt/conda/lib/python3.12/site-packages/sagemaker/model_monitor/model_monitoring.py", line 1594, in _create_monitoring_schedule_from_job_definition
    self.sagemaker_session.sagemaker_client.create_monitoring_schedule(
  File "/opt/conda/lib/python3.12/site-packages/botocore/client.py", line 569, in _api_call
    return self._make_api_call(operation_name, kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/botocore/client.py", line 1023, in _make_api_call
    raise error_class(parsed_response, operatio

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:4                                                                                    │
│                                                                                                  │
│    1 #Create schedule                                                                            │
│    2 schedule_name_xgb_dq = "xgb-data-quality-schedule"                                          │
│    3                                                                                             │
│ ❱  4 dq_monitor.create_monitoring_schedule(                                                      │
│    5 │   monitor_schedule_name=schedule_name_xgb_dq,                                             │
│    6 │   endpoint_input=xgb_endpoint_name,                                                       │
│    7 │   output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/data-quality/xgb",                    │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model_monitor/model_monitoring.py:2050 in      │
│ create_monitoring_schedule                                                                       │
│                                                                                                  │
│   2047 │   │                                                                                     │
│   2048 │   │   # create schedule                                                                 │
│   2049 │   │   try:                                                                              │
│ ❱ 2050 │   │   │   self._create_monitoring_schedule_from_job_definition(                         │
│   2051 │   │   │   │   monitor_schedule_name=monitor_schedule_name,                              │
│   2052 │   │   │   │   job_definition_name=new_job_definition_name,                              │
│   2053 │   │   │   │   schedule_cron_expression=schedule_cron_expression,                        │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/sagemaker/model_monitor/model_monitoring.py:1594 in      │
│ _create_monitoring_schedule_from_job_definition                                                  │
│                                                                                                  │
│   1591 │   │   # config key MONITORING_SCHEDULE_INTER_CONTAINER_ENCRYPTION_PATH here             │
│   1592 │   │   # because no MonitoringJobDefinition is set for this call                         │
│   1593 │   │                                                                                     │
│ ❱ 1594 │   │   self.sagemaker_session.sagemaker_client.create_monitoring_schedule(               │
│   1595 │   │   │   MonitoringScheduleName=monitor_schedule_name,                                 │
│   1596 │   │   │   MonitoringScheduleConfig=monitoring_schedule_config,                          │
│   1597 │   │   │   Tags=all_tags or [],                                                          │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:569 in _api_call                      │
│                                                                                                  │
│    566 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    567 │   │   │   │   )                                                                         │
│    568 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  569 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    570 │   │                                               

In [45]:
#Check status of endpoint
sm_client = boto3.client('sagemaker')
response = sm_client.describe_endpoint(EndpointName=xgb_endpoint_name)
print("Status:", response['EndpointStatus'])
print("Last heartbeat:", response.get('LastHeartbeatTimestamp', 'N/A'))

Status: InService
Last heartbeat: N/A


In [47]:
# Create predictor
xg_predictor = Predictor(
    endpoint_name=xgb_endpoint_name,
    sagemaker_session=session,
)

# Load test data (scaled)
baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv")
test_data = baseline_df.drop("target", axis=1).iloc[:100]  # Send 100 samples

print(f"Sending {len(test_data)} predictions to generate monitoring data...")

# Send predictions
csv_payload = test_data.to_csv(header=False, index=False)
response = xg_predictor.predict(
    data=csv_payload,
    initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
)

print("✅ Predictions sent!")
print("Waiting for data capture to write to S3 (this takes a few minutes)...")
time.sleep(300)  #Wait a few minutes for data capture

Sending 100 predictions to generate monitoring data...
✅ Predictions sent!
Waiting for data capture to write to S3 (this takes a few minutes)...


### Set up a Model Quality Moniter

In [51]:
# Get predictions (batch all rows)
X_baseline = baseline_df.drop('target', axis=1)
csv_payload = X_baseline.to_csv(header=False, index=False)

# Get predictions from endpoint
response = xg_predictor.predict(
    data=csv_payload,
    initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
)

# Parse CSV predictions (format: prob1,prob2,...,prob7\n per row)
predictions_text = response.decode('utf-8').strip().split('\n')
prob_matrix = np.array([[float(x) for x in line.split(',') if x] for line in predictions_text])
pred_labels = np.argmax(prob_matrix, axis=1)

# Create MQ baseline dataframe
mq_baseline_df = pd.DataFrame({
    'prediction': pred_labels,
    'ground_truth_label': baseline_df['target'].values
})

# Save using s3_client instead
mq_baseline_key = "models/benchmarks/mq_baseline.csv"
csv_buffer = StringIO()
mq_baseline_df.to_csv(csv_buffer, index=False, header=True)

s3_client.put_object(
    Bucket=bucket,
    Key=mq_baseline_key,
    Body=csv_buffer.getvalue()
)

mq_baseline_uri = f"s3://{bucket}/{mq_baseline_key}"
print(f"MQ baseline: {mq_baseline_uri} ({len(mq_baseline_df)} rows)")

MQ baseline: s3://sagemaker-us-east-1-418418308994/models/benchmarks/mq_baseline.csv (10242 rows)


In [48]:
# Create Model Quality Monitor
mq_monitor = ModelQualityMonitor(
    role=role, 
    instance_count=1, 
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20, 
    max_runtime_in_seconds=3600, 
    sagemaker_session=session,
)

# Define problem type and constraints
model_quality_baseline_uri = f"s3://{bucket}/{prefix}/monitoring/mq-baseline"

INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.


In [56]:
#Suggest baseline for model quality
mq_monitor.suggest_baseline(
    baseline_dataset=mq_baseline_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=model_quality_baseline_uri,
    problem_type='MulticlassClassification',
    inference_attribute='prediction',  # Must match column name in CSV
    ground_truth_attribute='ground_truth_label',  # Must match column name in CSV
    wait=True,
    logs=False
)

print("✅ Model Quality baseline created!")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-06-02-39-36-556


...........................................................!✅ Model Quality baseline created!


In [ ]:
# Check baseline statistics
mq_stats_uri = mq_monitor.latest_baselining_job.baseline_statistics.file_name
mq_constraints_uri = mq_monitor.latest_baselining_job.suggested_constraints.file_name

print(f"Stats: {mq_stats_uri}")
print(f"Constraints: {mq_constraints_uri}")

In [ ]:
from datetime import datetime, timedelta

# Fake production ground truth (uses your train labels)
ground_truth_key = f"ground-truth-{datetime.now().strftime('%Y%m%d-%H')}.jsonl"
fake_gt = []

for i, gt_label in enumerate(mq_baseline_df['ground_truth_label']):  # CHANGED from 'target'
    fake_gt.append(json.dumps({
        "groundTruthData": {
            "data": str(int(gt_label)),
            "encoding": "CSV"
        },
        "eventMetadata": {
            "eventId": f"lab-{i}",
            "inferenceTime": (datetime.now() - timedelta(hours=1)).isoformat()
        },
        "eventVersion": "0"
    }))

# Upload ground truth
s3_resource = boto3.resource('s3')
s3_resource.Object(bucket, f"{prefix}/ground-truth/{ground_truth_key}").put(Body='\n'.join(fake_gt))

print(f"Ground truth uploaded: s3://{bucket}/{prefix}/ground-truth/{ground_truth_key}")

# Get constraints from the baseline job
mq_constraints = mq_monitor.latest_baselining_job.suggested_constraints

# Create monitoring schedule
mq_monitor.create_monitoring_schedule(
    monitor_schedule_name="xgb-model-quality-schedule",
    endpoint_input=xgb_endpoint_name,
    ground_truth_input=f"s3://{bucket}/{prefix}/ground-truth/",
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/model-quality/xgb",
    constraints=mq_constraints,
    schedule_cron_expression=CronExpressionGenerator.daily(),
    enable_cloudwatch_metrics=True,
)

print("✅ Full model quality monitoring active!")

### Infrastructure Monitors

In [ ]:
# Infrastructure monitoring - CloudWatch alarms for XGBoost endpoint
endpoint_metric_dimensions = [
    {"Name": "EndpointName", "Value": xgb_endpoint_name},  # CHANGED from lr_endpoint_name
]

# Alarm on high 5xx error rate
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-5XX-Errors-High",  # CHANGED name
    AlarmDescription="5XX error rate for XGBoost endpoint above threshold",  # Description
    Namespace="AWS/SageMaker",
    MetricName="ModelLatency",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Average",
    Period=300,  # 5 minutes
    EvaluationPeriods=2,
    Threshold=10000.0,  # 10 seconds in milliseconds
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False, 
)

print("✅ CloudWatch alarm created for XGBoost endpoint latency")

#Add alarm for invocation errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-Invocation-Errors",
    AlarmDescription="Invocation errors for XGBoost endpoint",
    Namespace="AWS/SageMaker",
    MetricName="ModelInvocationErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,
    EvaluationPeriods=1,
    Threshold=5.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)

print("✅ CloudWatch alarm created for XGBoost endpoint errors")

In [ ]:
sns_client = boto3.client('sns', region_name='us-east-1')

# Create SNS topic
response = sns_client.create_topic(Name='SageMaker-Endpoint-Alerts')
topic_arn = response['TopicArn']

print(f"SNS Topic created: {topic_arn}")

### CloudWatch Monitoring Dashboard

In [ ]:
import json

dashboard_name = "SageMaker-ML-Benchmarks"

dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Endpoint – Invocations & Latency",  # CHANGED
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", xgb_endpoint_name],  # CHANGED
                    [".", "ModelLatency", ".", "."],
                ],
                "stacked": False,
                "stat": "Average",
                "period": 60,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,  # CHANGED - put next to first widget
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Endpoint – Errors",  # NEW widget
                "metrics": [
                    ["AWS/SageMaker", "ModelInvocationErrors", "EndpointName", xgb_endpoint_name],
                    [".", "Invocation4XXErrors", ".", "."],
                    [".", "Invocation5XXErrors", ".", "."],
                ],
                "stacked": False,
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Data Quality Violations",  # CHANGED
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "DataQualityViolation",
                        "MonitoringSchedule",
                        "xgb-data-quality-schedule",  # CHANGED - update to your actual schedule name
                    ],
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,  # CHANGED
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Model Quality Violations",  # CHANGED
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "ModelQualityViolation",
                        "MonitoringSchedule",
                        "xgb-model-quality-schedule",  # CHANGED
                    ],
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
    ]
}

cw_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body),
)

print(f"✅ CloudWatch Dashboard created: {dashboard_name}")
print(f"View at: https://console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}")

### Generate Model & Data Reports in SageMaker

In [ ]:
#Generate a bias report


In [ ]:
fs = s3fs.S3FileSystem()

latest_dq_output_prefix = f"{bucket}/{prefix}/monitoring/data-quality/lr"
dq_reports = fs.ls(latest_dq_output_prefix)
dq_reports

# Example: load latest constraint violations
violations_path = [p for p in dq_reports if p.endswith("constraint_violations.json")][-1]

with fs.open(violations_path, "r") as f:
    dq_violations = json.load(f)

dq_violations


# Clean Up

In [ ]:
import boto3
import time

sm_client = boto3.client('sagemaker')
endpoint_name = "xgb-benchmark-endpoint"

#List & delete ALL monitoring schedules
response = sm_client.list_monitoring_schedules(MaxResults=50)
for sched in response['MonitoringScheduleSummaries']:
    sched_name = sched['MonitoringScheduleName']
    try:
        sm_client.delete_monitoring_schedule(MonitoringScheduleName=sched_name)
        print(f"Deleted schedule: {sched_name}")
    except Exception as e:
        print(f"Skip {sched_name}: {e}")

time.sleep(30)  # Wait for deletion

#Delete endpoint config
try:
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
    print(f"Deleted config: {endpoint_name}")
except Exception as e:
    print(f"Config: {e}")

#Delete endpoint
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f"Endpoint deletion started: {endpoint_name}")
except Exception as e:
    print(f"Endpoint: {e}")

print("⏳ Wait 2-5min, check SageMaker Console → Endpoints")